In [10]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.metrics import classification_report

In [39]:
# Read the data file and assign column names and display first 10 rows
source_df = pd.read_csv("data/sample_orig_2021.txt", delimiter = "|")
source_df.columns = ['credit_score','first_payment_date','first_time_homebuyer_flag','maturity_date','msa','mortgage_insurance_percentage',
                     'num_units','occupancy_status','cltv_score','dti','upb','ltv','interest_rate','channel','ppm_flag','amortization_type','state',
                     'property_type','postal_code','loan_sequence_number','loan_purpose','loan_term','num_borrowers','seller_name',
                     'servicer_name','super_conforming_flag','loan_sequence_number','p_indicator','harp_indicator','pvm','io_indicator',
                     'mortgage_ins_cancel_indicator'
]
display(source_df.head(10))

,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mortgage_insurance_percentage,num_units,occupancy_status,cltv_score,dti,...,num_borrowers,seller_name,servicer_name,super_conforming_flag,loan_sequence_number,p_indicator,harp_indicator,pvm,io_indicator,mortgage_ins_cancel_indicator
0,746,202103,N,205102,NaN,30,1,P,95,997,...,1,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,N
1,710,202103,N,204102,NaN,0,1,P,65,50,...,1,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7
2,722,202103,N,205102,16984.0,25,1,P,89,40,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,1,N,N
3,761,202103,N,205102,29020.0,0,1,P,56,50,...,1,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7
4,200,202103,N,203602,NaN,0,1,P,37,999,...,1,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7
5,799,202103,N,205102,17140.0,0,1,P,64,18,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,1,N,7
6,785,202103,N,205102,33340.0,0,1,P,75,30,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,1,N,7
7,692,202103,N,203602,41180.0,0,1,P,70,191,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7
8,688,202103,N,204102,NaN,25,1,P,95,42,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,N
9,797,202104,N,205103,20260.0,0,1,P,48,35,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7


In [40]:
# Label data for default based on following criteria
def simulate_default(row):
    if (row['credit_score'] <= 580) or (row['dti'] >= 60) or (row['cltv_score'] >= 95) or (row['interest_rate'] >= 9):
        label = 'True'
    else:
        label = 'False'
    return label

source_df['label'] = source_df.apply(simulate_default, axis = 1)
display(source_df.head(10))

,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mortgage_insurance_percentage,num_units,occupancy_status,cltv_score,dti,...,seller_name,servicer_name,super_conforming_flag,loan_sequence_number,p_indicator,harp_indicator,pvm,io_indicator,mortgage_ins_cancel_indicator,label
0,746,202103,N,205102,NaN,30,1,P,95,997,...,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,N,True
1,710,202103,N,204102,NaN,0,1,P,65,50,...,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7,False
2,722,202103,N,205102,16984.0,25,1,P,89,40,...,Other sellers,Other servicers,NaN,NaN,9,NaN,1,N,N,False
3,761,202103,N,205102,29020.0,0,1,P,56,50,...,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7,False
4,200,202103,N,203602,NaN,0,1,P,37,999,...,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7,True
5,799,202103,N,205102,17140.0,0,1,P,64,18,...,Other sellers,Other servicers,NaN,NaN,9,NaN,1,N,7,False
6,785,202103,N,205102,33340.0,0,1,P,75,30,...,Other sellers,Other servicers,NaN,NaN,9,NaN,1,N,7,False
7,692,202103,N,203602,41180.0,0,1,P,70,191,...,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7,True
8,688,202103,N,204102,NaN,25,1,P,95,42,...,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,N,True
9,797,202104,N,205103,20260.0,0,1,P,48,35,...,Other sellers,Other servicers,NaN,NaN,9,NaN,2,N,7,False


In [46]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Feature engineering - Select features required for ML Model
features = ['credit_score', 'cltv_score', 'dti', 'interest_rate','label']
ml_df = source_df[features]
display(ml_df.head(10))

# Example preprocessing steps (adapt to your specific features)
# Assuming 'target' is your loan default variable (1 for default, 0 for no default)
cleaned_df = ml_df.dropna()
X = cleaned_df.drop('label', axis = 1)
y = cleaned_df['label']

# Handle categorical features (example with LabelEncoder, but OneHotEncoder might be preferred)
for column in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X[column] = le.fit_transform(X[column])

# Handle numerical features (example with StandardScaler)
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Address class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Example using Logistic Regression
model = LogisticRegression(random_state=42, solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Evaluate performance
print(classification_report(y_test, y_pred))
print(f'ROC AUC: {roc_auc_score(y_test, y_proba)}')


,credit_score,cltv_score,dti,interest_rate,label
0,746,95,997,2.875,True
1,710,65,50,2.875,False
2,722,89,40,2.875,False
3,761,56,50,2.750,False
4,200,37,999,2.625,True
5,799,64,18,2.875,False
6,785,75,30,2.875,False
7,692,70,191,2.875,True
8,688,95,42,2.500,True
9,797,48,35,2.750,False


              precision    recall  f1-score   support

       False       1.00      0.98      0.99      8987
        True       0.98      1.00      0.99      9010

    accuracy                           0.99     17997
   macro avg       0.99      0.99      0.99     17997
weighted avg       0.99      0.99      0.99     17997

ROC AUC: 0.997529518220115


In [48]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib # To save and load the model and preprocessors

# Create a sample new data point (replace with your actual new data)
new_data = pd.DataFrame([{
    'credit_score': 722,
    'cltv_score': 89,
    'dti': 40,
    'interest_rate': 2.875
}])

# Apply the same StandardScaler as during training
numerical_cols = new_data.select_dtypes(include=['int64', 'float64']).columns
new_data[numerical_cols] = scaler.transform(new_data[numerical_cols])

# Make predictions on the preprocessed new data
prediction = model.predict(new_data)
prediction_proba = model.predict_proba(new_data)[:, 1] # Probability of default

print(f"Predicted default status: {prediction[0]}")
print(f"Probability of default: {prediction_proba[0]:.4f}")

if prediction[0] == 'True':
    print("Advice here")
else:
    print("All good - Keep it up!")
    

Predicted default status: False
Probability of default: 0.4058
All good - Keep it up!


In [ ]:
# Test Data Inpit:
# this results true - 688	95	42	2.500
# this results in false - 761	56	50	2.750
# This result in false - 722	89	40	2.875